instance du cycle de credit : on admet que le client dispose d'un historique, ou est actif

        - Observation WINDOW : 12 MOIS , car le cycle de fonctionnement du pret immobilier est saisonier
        - 90 DPD = defaut
        - Diversite des etat de pret : early , actif , default ( donc que le balance code couvre la mojorite des cas )

In [2]:
import pandas as pd



# Pour le svcg

In [3]:
svcg = pd.read_csv('/Users/macbookpro/platform/Backend/data/working/svcg_v01.csv')

/var/folders/gy/2mbpv0kx5jz22fry28cqlnkc0000gn/T/ipykernel_80940/3499056830.py:1: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  svcg = pd.read_csv('/Users/macbookpro/platform/Backend/data/working/svcg_v01.csv')


Suppression des colonnes non utiles

GROUPE 1 — RETARD

FRÉQUENCE

    Question   : combien de fois le client est-il en retard ?
    Colonne    : CURRENT_LOAN_DELINQUENCY_STATUS
    Opération  : count(DPD > 0) / nb_mois_total
    C ombinaisons :
        - Fréquence × Profondeur max → souvent en retard ET va loin
        - Fréquence × Récupération  → souvent en retard mais se rattrape
        - Fréquence × Tendance      → retards fréquents ET croissants

SÉVÉRITÉ

     Question   : en moyenne jusqu'où vont les retards ?
     Colonne    : CURRENT_LOAN_DELINQUENCY_STATUS
    Opération  : mean(DPD)

TENDANCE

    Question   : les retards augmentent-ils ou diminuent-ils ?
    Colonne    : CURRENT_LOAN_DELINQUENCY_STATUS
    Opération  : slope(DPD) sur la fenêtre
    Sens       : pente positive = dégradation / négative = amélioration

RÉCUPÉRATION

    Question   : le client parvient-il à revenir à 0 DPD ?
    Colonne    : CURRENT_LOAN_DELINQUENCY_STATUS
    Opération  : nb_mois_avant_retour_0 après chaque épisode de retard

PROFONDEUR MAX

    Question   : quel est le pire DPD jamais atteint ?
    Colonne    : CURRENT_LOAN_DELINQUENCY_STATUS
    Opération  : max(DPD)
    Combinaison: n_fois_max × max → récidivisme au niveau extrême
    Sens       : capture les habitués du pire DPD

GROUPE 2 — CAPITAL

NIVEAU

    Question   : quelle part du capital reste due ?
    Colonnes   : CURRENT_ACTUAL_UPB / UPB_origination
    Opération  : ratio
    Sens       : plus le ratio est élevé, plus l'exposition de la banque est grande

PROGRESSION

    Question   : le capital baisse-t-il régulièrement ?
    Colonne    : CURRENT_ACTUAL_UPB
    Opération  : std(delta_UPB) sur la fenêtre
    Sens       : forte variance = progression irrégulière = signal de stress

ÉCART AU PLAN

    Question   : est-on en retard sur le remboursement théorique ?
    Colonnes   : CURRENT_ACTUAL_UPB + LOAN_AGE + REMAINING_MONTHS_TO_LEGAL_MATURITY
    Opération  : UPB_theorique - CURRENT_ACTUAL_UPB au point d'observation
    Sens       : écart positif = client en retard sur son plan

ANTICIPATION

    Question   : rembourse-t-il plus que prévu ?
    Colonnes   : CURRENT_ACTUAL_UPB + CURRENT_INTEREST_RATE
    Opération  : nb_mois_remboursement_superieur_plan / nb_mois_total
    Sens       : ratio élevé = client financièrement solide

NOTE

      INTEREST_BEARING_UPB + CURRENT_NON_INTEREST_BEARING_UPB
    → affinent les angles 3 et 4
    → indiquent quelle part du capital génère réellement des intérêts

Definition du statu de defaut:



In [4]:
svcg.columns

Index(['LOAN_SEQUENCE_NUMBER', 'MONTHLY_REPORTING_PERIOD',
       'CURRENT_ACTUAL_UPB', 'CURRENT_LOAN_DELINQUENCY_STATUS', 'LOAN_AGE',
       'REMAINING_MONTHS_TO_LEGAL_MATURITY', 'MODIFICATION_FLAG',
       'ZERO_BALANCE_CODE', 'ZERO_BALANCE_EFFECTIVE_DATE',
       'CURRENT_INTEREST_RATE', 'CURRENT_NON_INTEREST_BEARING_UPB',
       'DUE_DATE_OF_LAST_PAID_INSTALLMENT', 'INTEREST_RATE_STEP_INDICATOR',
       'ESTIMATED_LTV', 'DELINQUENCY_DUE_TO_DISASTER',
       'BORROWER_ASSISTANCE_STATUS_CODE', 'INTEREST_BEARING_UPB', 'VINTAGE'],
      dtype='object')